# Logistic regression 
## Introduction


## Definitions
### The maths


### Logit transformation


### Maximum likelihood estimation


### Interpretation


## Assumptions
### Linearity
### Independence of errors
### Homoscedasticity
### Normality of errors
### No perfect multicollinearity


## Real-world example
### Getting the data


In [ ]:
import pandas as pd

# Load the dataset from the URL of the MOOC
data = pd.read_csv(
    # "https://lms.fun-mooc.fr/c4x/Paris11/15001/asset/smp2.csv",
    "./data/smp2.csv",  # If workding from the GitHub repository
    delimiter=';')

print("First rows of the dataframe:")
print(data.head())

### Building the logistic regression model


In [ ]:
import statsmodels.api as sm

# Drop rows with missing values for `suicide.hr` and `abus`
data_logit = data[
    ['suicide.hr', 'abus']].copy().dropna()

# Define the dependent variable (y) and independent variables (X)
y = data_logit['suicide.hr']
X = data_logit[['abus']]

# Add a constant term to the independent variable matrix
X = sm.add_constant(X)

# Fit the logistic regression model using MLE
model_sm = sm.Logit(y, X)
result_sm = model_sm.fit()

# Print the model summary
print(result_sm.summary())

In [ ]:
import statsmodels.formula.api as smf

# Define the model formula using patsy syntax with the interaction term
formula = "Q('suicide.hr') ~ abus"

# Fit the MLE model using the formula and data
model = smf.logit(formula, data=data_logit)

# Fit the model
result = model.fit()

# Print the regression result with a slightly different method
print("Logistic regression with a binary independent variable:")
print(result.summary2())

In [ ]:
import numpy as np

# Calculate the odds and probability for the intercept
intercept_odds = np.exp(result.params.loc['Intercept'])
intercept_prob = intercept_odds / (1 + intercept_odds)  # p=odds/(1+odds)

# Calculate the odds ratio and confidence interval for *abus*
abus_coeff_or = np.exp(result.params.loc['abus'])
abus_coeff_ci = np.exp(result.conf_int().loc['abus'].to_numpy())

# Print the odds and probability for the intercept
print("Intercept:")
print(f" Baseline log-odds (abus=0) = {result.params.loc['Intercept']:.3f}")
print(f" Baseline odds (abus=0) = {intercept_odds:.3f}")
print(f" Probability (abus=0) = {intercept_prob:.3f}")
print()
# Print the odds ratio, and confidence interval for *abus*
print("abus:")
print(f" Log-odds = {result.params.loc['abus']:.3f}")
print(f" Odds ratio = {abus_coeff_or:3.3f}")
print(f" CI of the OR: ", abus_coeff_ci.round(2))

### Contingency table


In [ ]:
# Create the contingency table
contingency_table = pd.crosstab(
    data_logit['abus'], data_logit['suicide.hr'])

# Print the contingency table
print("Contingency table:")
print(contingency_table)

In [ ]:
# Calculate the odds ratio and confidence interval
table2x2_contingency = sm.stats.Table2x2(contingency_table)
print(table2x2_contingency.summary())

### Logistic regression with a continuous variable


In [ ]:
# Drop rows with missing values for `suicide.hr` and `age`
data_age = data[['suicide.hr', 'age']].copy().dropna()

# Define the model formula using patsy syntax
formula_age = "Q('suicide.hr') ~ age"

# Fit the MLE model using the formula and data
model_age = smf.logit(formula_age, data=data_age)
result_age = model_age.fit()

# Print the regression result summary
print("Logistic regression with a continuous independent variable:")
print(result_age.summary2())

In [ ]:
# Calculate the odds ratio and confidence interval for *age*
age_coeff_or = np.exp(result_age.params.loc['age'])
age_coeff_ci = np.exp(result_age.conf_int().loc['age'].to_numpy())

# Print the odds ratio, and confidence interval for *age*
print("age:")
print(f" Log-odds = {result_age.params.loc['age']:.3f}")
print(f" Odds ratio = {age_coeff_or:3.3f}")
print(f" CI of the OR: ", age_coeff_ci.round(2))

### Predictions


In [ ]:
# Example data for prediction
age_new = pd.DataFrame({'age': [20, 30, 40, 50]})

# Predict probabilities
predicted_probabilities = result_age.predict(exog=age_new)

# Create a Pandas Series with age as index
predicted_probabilities = pd.Series(
    predicted_probabilities.values,
    name='prediction',
    index=age_new['age'])

print("Predicted high suicide probabilities by age:")
print(predicted_probabilities)

### Visualization


In [ ]:

import matplotlib.pyplot as plt

# Generate age values for plotting
age_new_range = np.linspace(data['age'].min(), data['age'].max(), 1000)
age_new_df = pd.DataFrame({'age': age_new_range})

# Predict probabilities
y_proba = result_age.predict(exog=age_new_df)

plt.figure(figsize=(6, 3))

plt.plot(
    age_new_range, y_proba,
    color='red', linewidth=2,
    label="Probability of high suicide risk")

# Plot original data points (age vs. suicide.hr)
plt.plot(
    data[data['suicide.hr'] == 0]['age'],
    data[data['suicide.hr'] == 0]['suicide.hr'],
    '+', label="suicide.hr = 0")
plt.plot(
    data[data['suicide.hr'] == 1]['age'],
    data[data['suicide.hr'] == 1]['suicide.hr'],
    '+', label="suicide.hr = 1")

plt.xlabel("Age")
plt.ylabel("Probability")
plt.legend(loc='center')
plt.axis([data['age'].min()-5, data['age'].max()+5, -0.02, 1.02])
plt.grid();

### Fitting a more complex model


In [ ]:
# Drop rows with missing values for relevant variables
data_interaction = data[
    ['suicide.hr', 'abus', 'discip', 'duree']].copy().dropna()

# Create the patsy formula using asterisk-notation
formula_interaction = "Q('suicide.hr') ~ abus + discip*duree"
#formula_interaction = """Q('suicide.hr') ~ abus 
#+ discip + duree + discip:duree"""

# Fit the logistic regression model using the Patsy formula
model_interaction = smf.logit(
    formula_interaction, data=data_interaction)
result_interaction = model_interaction.fit()

# Print the model summary
print("Logistic regression with interaction term:")
print(result_interaction.summary2())

In [ ]:
# Calculate the odds ratio and confidence interval for *duree*
duree_coeff_or = np.exp(result_interaction.params.loc['duree'])

# Print the odds ratio, and confidence interval for *duree*
print("duree:")
print(f" Log-odds = {result_interaction.params.loc['duree']:.3f}")
print(f" Odds ratio = {duree_coeff_or:3.3f}")

### Conclusion
